# Download the full THINGS-EEG2 dataset (all 10 subjects)

Fetches the **entire** THINGS-EEG2 dataset used by this pipeline and arranges it into the
exact layout the downstream steps expect:

* **Raw EEG**, all 10 subjects x 4 sessions -> `subXX_raw/sub-XX/ses-0S/raw_eeg_{train,test}.npy`
* **Stimulus images** -> `images/training_images/`, `images/test_images/`
* **Image metadata** -> `image_metadata.npy`

### Source
The official OSF **Raw EEG** component ([osf.io/3jk45](https://osf.io/3jk45/) -> `crxs4`) is empty
via the OSF API, so raw cannot be pulled from OSF. A complete mirror of the raw `.npy` data
(the same `{raw_eeg_data, ch_names, ch_types, sfreq}` dicts `000_preprocessing_eeg.ipynb`
reads) is hosted on the Hugging Face Hub:
[**gasparyanartur/things-eeg2**](https://huggingface.co/datasets/gasparyanartur/things-eeg2).
This notebook downloads from there.

### Running it unattended (session off)
A notebook only runs while its kernel is alive. To let it finish after you disconnect, run it
**headless** from a terminal instead of cell-by-cell:

```bash
cd ~/things_eeg
nohup jupyter nbconvert --to notebook --execute --inplace \
  --ExecutePreprocessor.timeout=-1 download_raw_eegdata.ipynb \
  > download.log 2>&1 &
# watch progress:  tail -f ~/things_eeg/download.log
```

`nohup ... &` keeps it running after logout; the download is **resumable**, so if it dies you
can re-launch the same command and it skips whatever is already complete.

### Size / disk
~14 GB per subject -> ~140 GB total. Files are moved (not copied) into place to avoid using
double the space. The cell below checks free disk before starting.

In [1]:
# Hugging Face Hub client (one-off install).
%pip install -q huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, shutil, zipfile

# --- configuration -------------------------------------------------------
HF_REPO   = "gasparyanartur/things-eeg2"      # dataset repo (raw .npy mirror + images)
ROOT      = os.path.expanduser("~/things_eeg")
STAGE     = os.path.join(ROOT, "hf_stage")    # staging dir on the SAME filesystem as ROOT
SUBJECTS  = list(range(1, 11))                # all 10 subjects; already-present files are skipped
N_SES     = 4                                 # sessions per subject
GET_IMAGES = True                             # also fetch + unzip the stimulus images

# Map local mode -> remote filename. Pipeline wants raw_eeg_train.npy;
# the mirror stores training as raw_eeg_training.npy (test matches).
MODE_FILES = {"train": "raw_eeg_training.npy", "test": "raw_eeg_test.npy"}
os.makedirs(STAGE, exist_ok=True)

free_gb = shutil.disk_usage(ROOT).free / 1e9
print(f"repo: {HF_REPO}")
print(f"root: {ROOT}")
print(f"subjects: {SUBJECTS}")
print(f"free disk: {free_gb:.0f} GB  (full raw set is ~140 GB; sub-01 may already be present)")

repo: gasparyanartur/things-eeg2
root: /home/feyzanur_mbb/things_eeg
subjects: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
free disk: 187 GB  (full raw set is ~140 GB; sub-01 may already be present)


## 1. Download all raw EEG

Each file is fetched with `hf_hub_download` (resumable) into `hf_stage/`, then **moved** into
`subXX_raw/sub-XX/ses-0S/` (renaming `training` -> `train`). A move on the same filesystem is
instant and uses no extra space. Files already in place are skipped, so re-running resumes.

In [3]:
from huggingface_hub import hf_hub_download

n_done = n_skip = 0
for sub in SUBJECTS:
    dest_root = os.path.join(ROOT, f"sub{sub:02d}_raw")
    for s in range(1, N_SES + 1):
        ses_dir = os.path.join(dest_root, f"sub-{sub:02d}", f"ses-{s:02d}")
        os.makedirs(ses_dir, exist_ok=True)
        for mode, remote_name in MODE_FILES.items():
            dest = os.path.join(ses_dir, f"raw_eeg_{mode}.npy")
            if os.path.exists(dest):
                n_skip += 1
                print(f"sub-{sub:02d} ses-{s:02d} {mode:5s}: present -> skip", flush=True)
                continue
            remote_path = f"raw-eeg/sub-{sub:02d}/ses-{s:02d}/{remote_name}"
            print(f"sub-{sub:02d} ses-{s:02d} {mode:5s}: downloading {remote_path} ...", flush=True)
            cached = hf_hub_download(HF_REPO, remote_path, repo_type="dataset", local_dir=STAGE)
            os.replace(cached, dest)          # move within same filesystem: instant, no copy
            n_done += 1
            print(f"   -> {dest}", flush=True)
print(f"\nRAW DONE. downloaded {n_done}, skipped {n_skip}.", flush=True)

sub-01 ses-01 train: present -> skip


sub-01 ses-01 test : present -> skip


sub-01 ses-02 train: present -> skip


sub-01 ses-02 test : present -> skip


sub-01 ses-03 train: present -> skip


...

sub-10 ses-04 train: downloading raw-eeg/sub-10/ses-04/raw_eeg_training.npy ...


   -> /home/feyzanur_mbb/things_eeg/sub10_raw/sub-10/ses-04/raw_eeg_train.npy


sub-10 ses-04 test : downloading raw-eeg/sub-10/ses-04/raw_eeg_test.npy ...


   -> /home/feyzanur_mbb/things_eeg/sub10_raw/sub-10/ses-04/raw_eeg_test.npy



RAW DONE. downloaded 68, skipped 12.


## 2. Download + unzip the stimulus images and metadata

`000_preprocessing_eeg.ipynb` and `02_extract_features.py` read the images from
`images/training_images/` and `images/test_images/`. This unzips them there and drops
`image_metadata.npy` at the project root. Skipped if the folders already exist.

In [4]:
if GET_IMAGES:
    img_dir = os.path.join(ROOT, "images")
    os.makedirs(img_dir, exist_ok=True)
    for zipname, outdir in [("training_images", "training_images"), ("test_images", "test_images")]:
        target = os.path.join(img_dir, outdir)
        if os.path.isdir(target) and os.listdir(target):
            print(f"{outdir}: already present -> skip", flush=True)
            continue
        print(f"{outdir}: downloading imgs/{zipname}.zip ...", flush=True)
        zpath = hf_hub_download(HF_REPO, f"imgs/{zipname}.zip", repo_type="dataset", local_dir=STAGE)
        print(f"{outdir}: extracting -> {img_dir} ...", flush=True)
        with zipfile.ZipFile(zpath) as zf:
            zf.extractall(img_dir)
        os.remove(zpath)                      # drop the zip after extraction
        print(f"{outdir}: done", flush=True)

    # image metadata
    meta_dest = os.path.join(ROOT, "image_metadata.npy")
    if not os.path.exists(meta_dest):
        cached = hf_hub_download(HF_REPO, "image_metadata.npy", repo_type="dataset", local_dir=STAGE)
        os.replace(cached, meta_dest)
        print("image_metadata.npy -> saved", flush=True)
    else:
        print("image_metadata.npy: present -> skip", flush=True)
else:
    print("GET_IMAGES is False -> skipping images", flush=True)

training_images: already present -> skip


test_images: already present -> skip


image_metadata.npy: present -> skip


## 3. Verify everything downloaded correctly

Loads one `.npy` per subject/session and reports shape / channels / sfreq (should be
64 channels incl. `stim`, sfreq 1000), and confirms the image folders exist. Any MISSING
line means a file didn't complete -- re-run cell 1 to fetch just those.

In [5]:
import numpy as np

missing = 0
for sub in SUBJECTS:
    dest_root = os.path.join(ROOT, f"sub{sub:02d}_raw")
    for s in range(1, N_SES + 1):
        for mode in MODE_FILES:
            p = os.path.join(dest_root, f"sub-{sub:02d}", f"ses-{s:02d}", f"raw_eeg_{mode}.npy")
            if not os.path.isfile(p):
                print(f"sub-{sub:02d} ses-{s:02d} {mode:5s}: MISSING")
                missing += 1
    # spot-check the first session's test file for this subject
    probe = os.path.join(dest_root, f"sub-{sub:02d}", "ses-01", "raw_eeg_test.npy")
    if os.path.isfile(probe):
        d = np.load(probe, allow_pickle=True).item()
        print(f"sub-{sub:02d}: ses-01 test raw_eeg_data {d['raw_eeg_data'].shape} "
              f"| {len(d['ch_names'])} ch | sfreq {d['sfreq']}")

for d in ["images/training_images", "images/test_images"]:
    full = os.path.join(ROOT, d)
    n = len(os.listdir(full)) if os.path.isdir(full) else 0
    print(f"{d}: {'OK' if n else 'MISSING'} ({n} entries)")

print(f"\n{'ALL FILES PRESENT' if missing == 0 else str(missing) + ' FILES MISSING'}", flush=True)

sub-01: ses-01 test raw_eeg_data (64, 1355160) | 64 ch | sfreq 1000


sub-02: ses-01 test raw_eeg_data (64, 1427880) | 64 ch | sfreq 1000


sub-03: ses-01 test raw_eeg_data (64, 1328880) | 64 ch | sfreq 1000


sub-04: ses-01 test raw_eeg_data (64, 1814360) | 64 ch | sfreq 1000


sub-05: ses-01 test raw_eeg_data (64, 1474740) | 64 ch | sfreq 1000


sub-06: ses-01 test raw_eeg_data (64, 1326660) | 64 ch | sfreq 1000


sub-07: ses-01 test raw_eeg_data (64, 1455480) | 64 ch | sfreq 1000


sub-08: ses-01 test raw_eeg_data (64, 1651180) | 64 ch | sfreq 1000


sub-09: ses-01 test raw_eeg_data (64, 1384040) | 64 ch | sfreq 1000


sub-10: ses-01 test raw_eeg_data (64, 1495400) | 64 ch | sfreq 1000
images/training_images: OK (1654 entries)
images/test_images: OK (200 entries)

ALL FILES PRESENT


## 4. Clean up staging

Removes the (now-empty) `hf_stage/` scratch dir and its download metadata.

In [6]:
if os.path.isdir(STAGE):
    shutil.rmtree(STAGE)
    print("removed", STAGE)
print("\nDONE. Next: run 000_preprocessing_eeg.ipynb per subject "
      "(set raw_root='subXX_raw', sub=XX) to build preprocessed_data/.")

removed /home/feyzanur_mbb/things_eeg/hf_stage

DONE. Next: run 000_preprocessing_eeg.ipynb per subject (set raw_root='subXX_raw', sub=XX) to build preprocessed_data/.
